#Cell 1 — Mount Drive & config

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob

DRIVE_ROOT   = "/content/drive/MyDrive/jetcobot_2026"
RESULTS_DIR  = f"{DRIVE_ROOT}/mast3r_resultat"
IMAGES_ZIP   = f"{DRIVE_ROOT}/plant3_db.zip"
OUTPUT_DIR   = f"{RESULTS_DIR}/gaussian_splatting_results2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_SH_DEGREE = 3
NUM_ITERS = 15000
TEST_EVERY = 8          # every 8th image held out for eval (official 3DGS convention)
ENABLE_POSE_REFINEMENT = True
ENABLE_EDGE_AWARE_LOSS = True   # upweights thin structures (stems/leaf edges)
LPIPS_WEIGHT = 0.0       # set >0 (e.g. 0.05) to add LPIPS into the training loss, slower per-step

Mounted at /content/drive


#Cell 2 — Install

In [2]:
!pip install -q gsplat plyfile open3d pytorch-msssim lpips

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 120.7 MB/s eta 0:00:00


#Cell 3 — Imports

In [3]:
import torch, torch.nn.functional as F
import numpy as np, cv2, shutil, time, json, math
from plyfile import PlyData, PlyElement
from pytorch_msssim import ssim as ssim_fn
import lpips
from gsplat import rasterization
from gsplat.strategy import DefaultStrategy

device = "cuda"
assert torch.cuda.is_available(), "Switch runtime to GPU (T4)."
torch.manual_seed(0)
SH_C0 = 0.28209479177387814

#Cell 4 — Unzip images

In [4]:
IMAGES_LOCAL = "/content/plant3_db"
if not os.path.exists(IMAGES_LOCAL):
    shutil.unpack_archive(IMAGES_ZIP, IMAGES_LOCAL)

candidates = [os.path.dirname(p) for p in glob.glob(f"{IMAGES_LOCAL}/**/*.png", recursive=True)]
IMAGES_DIR = max(set(candidates), key=candidates.count)
image_paths = sorted(glob.glob(f"{IMAGES_DIR}/*.png"))
print(f"Found {len(image_paths)} images in {IMAGES_DIR}")

Found 139 images in /content/plant3_db/plant3_back


#Cell 5 — Load MASt3R poses/intrinsics/init cloud

In [5]:
poses_c2w = np.load(f"{RESULTS_DIR}/poses_c2w.npy")
intrinsics = np.load(f"{RESULTS_DIR}/intrinsics.npy")
N = poses_c2w.shape[0]
assert len(image_paths) == N, f"{len(image_paths)} images vs {N} poses — check ordering."
if intrinsics.ndim == 2:
    intrinsics = np.repeat(intrinsics[None], N, axis=0)

plydata = PlyData.read(f"{RESULTS_DIR}/cleaned.ply")
v = plydata['vertex']
init_xyz = np.stack([v['x'], v['y'], v['z']], axis=1).astype(np.float32)
if all(k in v.data.dtype.names for k in ('red','green','blue')):
    init_rgb = np.stack([v['red'], v['green'], v['blue']], axis=1).astype(np.float32) / 255.0
else:
    init_rgb = np.ones_like(init_xyz) * 0.5
print("Init points:", init_xyz.shape, "| Images:", N)

Init points: (1460815, 3) | Images: 139


#Cell 6 — Build tensors + train/test split

In [6]:
imgs = []
for p in image_paths:
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    imgs.append(im)
H, W = imgs[0].shape[:2]
imgs = torch.tensor(np.stack(imgs), device=device)

c2w = torch.tensor(poses_c2w, dtype=torch.float32, device=device)
viewmats_orig = torch.linalg.inv(c2w)
Ks = torch.tensor(intrinsics, dtype=torch.float32, device=device)

all_idx = np.arange(N)
test_idx = all_idx[::TEST_EVERY]
train_idx = np.setdiff1d(all_idx, test_idx)
print(f"Train views: {len(train_idx)} | Test views: {len(test_idx)}")

Train views: 121 | Test views: 18


#Cell 7 — Utility functions (PCA init, rotation->quaternion, SH conversion)

In [7]:
from scipy.spatial import cKDTree
from scipy.spatial.transform import Rotation as Rot

def rgb_to_sh(rgb):
    return (rgb - 0.5) / SH_C0

def pca_anisotropic_init_vectorized(xyz, k=16, batch_size=50000):
    """
    Vectorized PCA-based anisotropic init.
    - cKDTree.query does the k-NN search for ALL points in one batched C call.
    - Covariance, eigendecomposition, and quaternion conversion are done
      in batches via NumPy/scipy (no per-point Python loop).
    """
    n = xyz.shape[0]
    print(f"Building KD-tree for {n} points...")
    tree = cKDTree(xyz)

    print(f"Querying {k}-NN for all points (batched)...")
    # query in one shot; cKDTree handles this efficiently even for 500k+ pts
    _, nn_idx = tree.query(xyz, k=k, workers=-1)   # (N, k)

    log_scales = np.zeros((n, 3), dtype=np.float32)
    quats = np.zeros((n, 4), dtype=np.float32)

    print("Computing batched PCA covariance + eigendecomposition...")
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        idx_batch = nn_idx[start:end]                     # (B, k)
        nbrs = xyz[idx_batch]                              # (B, k, 3)
        centered = nbrs - nbrs.mean(axis=1, keepdims=True)  # (B, k, 3)

        # batched covariance: (B, 3, 3)
        cov = np.einsum('bki,bkj->bij', centered, centered) / k

        # batched eigendecomposition (ascending eigenvalues), vectorized over batch dim
        eigval, eigvec = np.linalg.eigh(cov)                # eigval (B,3), eigvec (B,3,3)
        eigval = np.clip(eigval, 1e-8, None)

        # ensure right-handed rotation matrices (det > 0), vectorized
        dets = np.linalg.det(eigvec)
        flip = dets < 0
        eigvec[flip, :, 0] *= -1

        log_scales[start:end] = np.log(np.sqrt(eigval)).astype(np.float32)

        # vectorized rotation-matrix -> quaternion via scipy (returns x,y,z,w)
        quat_xyzw = Rot.from_matrix(eigvec).as_quat()
        # reorder to (w,x,y,z) to match gsplat convention
        quats[start:end] = quat_xyzw[:, [3, 0, 1, 2]].astype(np.float32)

        if start % (batch_size * 4) == 0:
            print(f"  processed {end}/{n} points")

    return log_scales, quats

#Cell 8 — Initialize Gaussians (degree-3 SH, PCA anisotropic init)

In [8]:
print("Running vectorized PCA-based anisotropic init...")
log_scales, quats_np = pca_anisotropic_init_vectorized(init_xyz, k=16)

n_pts = init_xyz.shape[0]
sh_dim = (MAX_SH_DEGREE + 1) ** 2   # 16 for degree 3

sh0 = rgb_to_sh(torch.tensor(init_rgb, device=device)).unsqueeze(1)          # (N,1,3)
shN = torch.zeros(n_pts, sh_dim - 1, 3, device=device)                       # (N,15,3)

splats = torch.nn.ParameterDict({
    "means":     torch.nn.Parameter(torch.tensor(init_xyz, device=device)),
    "scales":    torch.nn.Parameter(torch.tensor(log_scales, device=device)),
    "quats":     torch.nn.Parameter(torch.tensor(quats_np, device=device)),
    "opacities": torch.nn.Parameter(torch.logit(torch.full((n_pts,), 0.1, device=device))),
    "sh0":       torch.nn.Parameter(sh0),
    "shN":       torch.nn.Parameter(shN),
}).to(device)
print("Initialized", n_pts, "Gaussians, SH degree up to", MAX_SH_DEGREE)

Running vectorized PCA-based anisotropic init...
Building KD-tree for 1460815 points...
Querying 16-NN for all points (batched)...
Computing batched PCA covariance + eigendecomposition...
  processed 50000/1460815 points
  processed 250000/1460815 points
  processed 450000/1460815 points
  processed 650000/1460815 points
  processed 850000/1460815 points
  processed 1050000/1460815 points
  processed 1250000/1460815 points
  processed 1450000/1460815 points
Initialized 1460815 Gaussians, SH degree up to 3


#Cell 9 — Per-param optimizers + LR scheduler (official 3DGS scheme)

In [9]:
scene_extent = float(np.linalg.norm(init_xyz.max(0) - init_xyz.min(0)))

param_lrs = {
    "means":     1.6e-4 * scene_extent,
    "scales":    5e-3,
    "quats":     1e-3,
    "opacities": 5e-2,
    "sh0":       2.5e-3,
    "shN":       2.5e-3 / 20,
}
optimizers = {
    name: torch.optim.Adam([{"params": splats[name], "lr": lr, "name": name}], eps=1e-15)
    for name, lr in param_lrs.items()
}

# exponential decay on the "means" (position) LR, official 3DGS style
means_lr_init, means_lr_final = param_lrs["means"], param_lrs["means"] * 0.01
def get_means_lr(step):
    t = np.clip(step / NUM_ITERS, 0, 1)
    return math.exp(math.log(means_lr_init) * (1 - t) + math.log(means_lr_final) * t)

#Cell 10 — Official densification/pruning strategy (gsplat DefaultStrategy)

In [10]:
strategy = DefaultStrategy(
    prune_opa=0.005,
    grow_grad2d=0.0002,
    grow_scale3d=0.01,
    prune_scale3d=0.1,
    refine_start_iter=500,
    refine_stop_iter=int(NUM_ITERS * 0.8),
    reset_every=3000,       # opacity reset cadence
    refine_every=100,       # split/clone/prune cadence
    absgrad=True,           # more accurate screen-space grad, recommended
    revised_opacity=False,
)
strategy_state = strategy.initialize_state(scene_scale=scene_extent)

#Cell 11 — Optional per-camera pose refinement

In [14]:
if ENABLE_POSE_REFINEMENT:
    pose_delta_r = torch.nn.Parameter(torch.zeros(N, 3, device=device))
    pose_delta_t = torch.nn.Parameter(torch.zeros(N, 3, device=device))
    pose_optimizer = torch.optim.Adam([pose_delta_r, pose_delta_t], lr=1e-4)

    def axis_angle_to_matrix(v):
        theta = v.norm(dim=-1, keepdim=True) + 1e-8
        axis = v / theta
        K = torch.zeros(v.shape[0], 3, 3, device=v.device)
        K[:,0,1], K[:,0,2] = -axis[:,2], axis[:,1]
        K[:,1,0], K[:,1,2] = axis[:,2], -axis[:,0]
        K[:,2,0], K[:,2,1] = -axis[:,1], axis[:,0]
        I = torch.eye(3, device=v.device).unsqueeze(0)
        return I + torch.sin(theta).unsqueeze(-1)*K + (1-torch.cos(theta)).unsqueeze(-1)*(K@K)

    def refine_viewmat(idx):
        R_delta = axis_angle_to_matrix(pose_delta_r[idx:idx+1])           # (1,3,3)
        R_new = R_delta @ viewmats_orig[idx:idx+1, :3, :3]                # (1,3,3)
        t_new = viewmats_orig[idx:idx+1, :3, 3] + pose_delta_t[idx:idx+1] # (1,3)

        top = torch.cat([R_new, t_new.unsqueeze(-1)], dim=-1)             # (1,3,4)
        bottom = viewmats_orig[idx:idx+1, 3:4, :]                         # (1,1,4), the [0,0,0,1] row
        vm = torch.cat([top, bottom], dim=1)                              # (1,4,4)
        return vm
else:
    def refine_viewmat(idx):
        return viewmats_orig[idx:idx+1]

#Cell 12 — Losses (L1 + D-SSIM + optional LPIPS, edge-aware weighting)

In [12]:
lpips_fn = lpips.LPIPS(net='vgg').to(device) if LPIPS_WEIGHT > 0 else None

def edge_weight_map(gt):
    gray = gt.mean(-1, keepdim=True).permute(2,0,1).unsqueeze(0)
    gx = F.conv2d(gray, torch.tensor([[[-1,0,1],[-2,0,2],[-1,0,1]]], dtype=torch.float32, device=device).unsqueeze(0), padding=1)
    gy = F.conv2d(gray, torch.tensor([[[-1,-2,-1],[0,0,0],[1,2,1]]], dtype=torch.float32, device=device).unsqueeze(0), padding=1)
    mag = torch.sqrt(gx**2 + gy**2)[0,0]
    return 1.0 + 3.0 * (mag / (mag.max() + 1e-8))   # thin structures get up to 4x weight

def compute_loss(pred, gt):
    if ENABLE_EDGE_AWARE_LOSS:
        w = edge_weight_map(gt).unsqueeze(-1)
        l1 = (torch.abs(pred - gt) * w).mean()
    else:
        l1 = torch.abs(pred - gt).mean()
    dssim = 1 - ssim_fn(pred.permute(2,0,1)[None], gt.permute(2,0,1)[None], data_range=1.0)
    loss = 0.8 * l1 + 0.2 * dssim
    if lpips_fn is not None:
        lp = lpips_fn((pred*2-1).permute(2,0,1)[None], (gt*2-1).permute(2,0,1)[None]).mean()
        loss = loss + LPIPS_WEIGHT * lp
    return loss

#Cell 13 — Training loop

In [16]:
start_time = time.time()
sh_degree_interval = max(NUM_ITERS // (MAX_SH_DEGREE + 1), 1)

for step in range(1, NUM_ITERS + 1):
    idx = int(np.random.choice(train_idx))
    sh_degree_cur = min(step // sh_degree_interval, MAX_SH_DEGREE)  # ramp SH 0->3

    for g in optimizers["means"].param_groups:
        g["lr"] = get_means_lr(step)

    colors = torch.cat([splats["sh0"], splats["shN"]], dim=1)
    viewmat = refine_viewmat(idx)
    render, alpha, info = rasterization(
      means=splats["means"],
      quats=splats["quats"] / splats["quats"].norm(dim=-1, keepdim=True),
      scales=torch.exp(splats["scales"]),
      opacities=torch.sigmoid(splats["opacities"]),
      colors=colors,
      viewmats=viewmat,
      Ks=Ks[idx:idx+1],
      width=W, height=H,
      sh_degree=sh_degree_cur,
      packed=False,
      absgrad=True,   # <-- required since strategy was built with absgrad=True
    )
    pred, gt = render[0], imgs[idx]
    loss = compute_loss(pred, gt)

    for opt in optimizers.values(): opt.zero_grad()
    if ENABLE_POSE_REFINEMENT: pose_optimizer.zero_grad()

    strategy.step_pre_backward(params=splats, optimizers=optimizers, state=strategy_state, step=step, info=info)
    loss.backward()

    for opt in optimizers.values(): opt.step()
    if ENABLE_POSE_REFINEMENT: pose_optimizer.step()

    strategy.step_post_backward(params=splats, optimizers=optimizers, state=strategy_state, step=step, info=info, packed=False)

    if step % 200 == 0:
        print(f"iter {step:5d} | loss {loss.item():.4f} | SH deg {sh_degree_cur} | gaussians {splats['means'].shape[0]}")
    if step % 2000 == 0:
        torch.save({k: v.detach().cpu() for k, v in splats.items()}, f"{OUTPUT_DIR}/checkpoint_iter{step}.pt")

train_time_sec = time.time() - start_time
print(f"Training done in {train_time_sec/60:.1f} min | Final gaussians: {splats['means'].shape[0]}")

iter   200 | loss 0.6401 | SH deg 0 | gaussians 1460815
iter   400 | loss 0.5887 | SH deg 0 | gaussians 1460815
iter   600 | loss 0.2683 | SH deg 0 | gaussians 1444383
iter   800 | loss 0.3054 | SH deg 0 | gaussians 1444105
iter  1000 | loss 0.1834 | SH deg 0 | gaussians 1446686
iter  1200 | loss 0.1972 | SH deg 0 | gaussians 1452062
iter  1400 | loss 0.1971 | SH deg 0 | gaussians 1458972
iter  1600 | loss 0.2201 | SH deg 0 | gaussians 1466674
iter  1800 | loss 0.2333 | SH deg 0 | gaussians 1477116
iter  2000 | loss 0.1949 | SH deg 0 | gaussians 1490195
iter  2200 | loss 0.2108 | SH deg 0 | gaussians 1503766
iter  2400 | loss 0.1866 | SH deg 0 | gaussians 1518971
iter  2600 | loss 0.2187 | SH deg 0 | gaussians 1534258
iter  2800 | loss 0.1719 | SH deg 0 | gaussians 1550127
iter  3000 | loss 0.1948 | SH deg 0 | gaussians 1565730
iter  3200 | loss 0.1777 | SH deg 0 | gaussians 1584119
iter  3400 | loss 0.2136 | SH deg 0 | gaussians 1600479
iter  3600 | loss 0.1479 | SH deg 0 | gaussians 

#Cell 14 — Evaluation (PSNR / SSIM / LPIPS / FPS / VRAM / model size)

In [17]:
lpips_eval = lpips.LPIPS(net='vgg').to(device) if lpips_fn is None else lpips_fn

psnrs, ssims, lpipss, fps_list = [], [], [], []
torch.cuda.reset_peak_memory_stats()
colors = torch.cat([splats["sh0"], splats["shN"]], dim=1)

with torch.no_grad():
    for idx in test_idx:
        t0 = time.time()
        render, _, _ = rasterization(
            means=splats["means"],
            quats=splats["quats"] / splats["quats"].norm(dim=-1, keepdim=True),
            scales=torch.exp(splats["scales"]),
            opacities=torch.sigmoid(splats["opacities"]),
            colors=colors, viewmats=viewmats_orig[idx:idx+1], Ks=Ks[idx:idx+1],
            width=W, height=H, sh_degree=MAX_SH_DEGREE, packed=False,
        )
        torch.cuda.synchronize()
        fps_list.append(1.0 / (time.time() - t0))

        pred, gt = render[0].clamp(0,1), imgs[idx]
        mse = F.mse_loss(pred, gt).item()
        psnrs.append(10 * math.log10(1.0 / max(mse, 1e-10)))
        ssims.append(ssim_fn(pred.permute(2,0,1)[None], gt.permute(2,0,1)[None], data_range=1.0).item())
        lpipss.append(lpips_eval((pred*2-1).permute(2,0,1)[None], (gt*2-1).permute(2,0,1)[None]).item())

vram_mb = torch.cuda.max_memory_allocated() / 1e6
n_gauss = splats["means"].shape[0]
model_size_mb = sum(p.numel() * 4 for p in splats.values()) / 1e6  # float32

metrics = {
    "PSNR": float(np.mean(psnrs)), "SSIM": float(np.mean(ssims)), "LPIPS": float(np.mean(lpipss)),
    "num_gaussians": int(n_gauss), "training_time_min": train_time_sec/60,
    "render_fps": float(np.mean(fps_list)), "peak_vram_mb": vram_mb, "model_size_mb": model_size_mb,
}
print(json.dumps(metrics, indent=2))
with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 88.0MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
{
  "PSNR": 10.359673904198495,
  "SSIM": 0.580427090326945,
  "LPIPS": 0.5091892133156458,
  "num_gaussians": 2194633,
  "training_time_min": 26.785409704844156,
  "render_fps": 53.23808501508995,
  "peak_vram_mb": 5085.369344,
  "model_size_mb": 517.933388
}


#Cell 15 — Export official 3DGS-format PLY

In [18]:
def save_official_3dgs_ply(path, splats, sh_degree):
    means = splats["means"].detach().cpu().numpy()
    scales = splats["scales"].detach().cpu().numpy()   # log-space, matches official format
    quats  = (splats["quats"] / splats["quats"].norm(dim=-1, keepdim=True)).detach().cpu().numpy()
    opac   = splats["opacities"].detach().cpu().numpy()  # logit-space, matches official format
    sh0    = splats["sh0"].detach().cpu().numpy().reshape(means.shape[0], -1)   # (N,3)
    shN    = splats["shN"].detach().cpu().numpy().reshape(means.shape[0], -1)   # (N, 3*(K-1))

    n = means.shape[0]
    attrs = ["x","y","z","nx","ny","nz"]
    attrs += [f"f_dc_{i}" for i in range(3)]
    attrs += [f"f_rest_{i}" for i in range(shN.shape[1])]
    attrs += ["opacity"] + [f"scale_{i}" for i in range(3)] + [f"rot_{i}" for i in range(4)]

    dtype = [(a, 'f4') for a in attrs]
    data = np.zeros(n, dtype=dtype)
    data['x'], data['y'], data['z'] = means[:,0], means[:,1], means[:,2]
    data['nx'] = data['ny'] = data['nz'] = 0.0
    for i in range(3): data[f'f_dc_{i}'] = sh0[:, i]
    for i in range(shN.shape[1]): data[f'f_rest_{i}'] = shN[:, i]
    data['opacity'] = opac
    for i in range(3): data[f'scale_{i}'] = scales[:, i]
    for i in range(4): data[f'rot_{i}'] = quats[:, i]

    PlyElement.describe(data, 'vertex')
    PlyData([PlyElement.describe(data, 'vertex')]).write(path)

final_ply = f"{OUTPUT_DIR}/plant_gaussian_splat_official.ply"
save_official_3dgs_ply(final_ply, splats, MAX_SH_DEGREE)
print("Saved official-format 3DGS ply to:", final_ply)
print("Results folder now contains:", os.listdir(RESULTS_DIR))

Saved official-format 3DGS ply to: /content/drive/MyDrive/jetcobot_2026/mast3r_resultat/gaussian_splatting_results2/plant_gaussian_splat_official.ply
Results folder now contains: ['intrinsics.npy', 'poses_c2w.npy', 'plant_pointcloud.ply', 'pointcloud_check.png', 'diagnostics_per_image.png', 'diagnostics_confidence_maps.png', 'cleaned.ply', 'gaussian_splatting_results', 'gaussian_splatting_results2']
